# Arsitrad — Gemma 4 Inference Demo
## End-to-End: RAG + Fine-tuned Gemma 4 2B → Gradio UI

**Runtime:** Colab with **T4 GPU** (free tier sufficient)

---

## Contents
1. Setup & Install
2. Download LoRA Adapters + ChromaDB
3. RAG Retrieval
4. Model Inference
5. Gradio UI

**Author:** Hanif — UNDIP Architecture / Harizco Swarm


## 1. Setup & Installation


In [2]:
#@title 1.2 — Install dependencies
!pip install -q --upgrade pip
!pip install -q \
    torch \
    transformers>=4.40.0 \
    peft>=0.10.0 \
    bitsandbytes \
    accelerate \
    scipy \
    chromadb>=0.4.0 \
    sentence-transformers>=2.5.0 \
    gradio>=4.0.0 \
    huggingface_hub \
    tqdm
!pip install -q unsloth
print('All dependencies installed!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.40.0 which is incompatible.
google-adk 1.28.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.40.0 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.40.0 which is incompatible.
opentelemetry-exp

In [3]:
#@title 1.3 — Imports & Config
import os
import torch
from pathlib import Path

BASE_DIR   = Path('/content/arsitrad')
LORA_DIR   = BASE_DIR / 'lora_adapters'
CHROMA_DIR = BASE_DIR / 'chroma_db'

RAG_CONFIG = {
    'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'collection_name': 'arsitrad_national_regulations',
    'top_k': 5,
    'min_similarity': 0.3,
}

MODEL_CONFIG = {
    'max_seq_length': 2048,
    'load_in_4bit': True,
}
print('Config ready.')


Config ready.


## 2. Download LoRA Adapters + ChromaDB


In [4]:
#@title 2.1 — Create directories
!mkdir -p /content/arsitrad/lora_adapters
!mkdir -p /content/arsitrad/chroma_db
print('Directories created.')


Directories created.


In [5]:
#@title 2.2 — Download LoRA adapters from GitHub Release
import os, subprocess

LORA_URL = 'https://github.com/arsitekberotot/arsitrad/releases/download/v1.0.0/arsitrad_finetuned_adapters.zip'
out = '/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters.zip'

if os.path.exists('/content/arsitrad/lora_adapters/arsitrad_finetuned_adapters'):
    print('LoRA adapters already exist, skipping download.')
else:
    print('Downloading LoRA adapters (~747 MB)...')
    subprocess.run(['wget', '-q', '--show-progress', '-O', out, LORA_URL], check=True)
    print('Unzipping...')
    subprocess.run(['unzip', '-q', '-o', out, '-d', '/content/arsitrad/lora_adapters/'], check=True)
    print('Done!')


Unzipping...
Done!


In [13]:
#@title 2.3 — Download and Fix ChromaDB Extraction
import os, subprocess, shutil

CHROMA_ZIP = '/content/arsitrad/chroma_db_release.zip'
TARGET_DIR = '/content/arsitrad/chroma_db'
TEMP_EXTRACT = '/content/arsitrad/temp_chroma'

# Clean up previous attempts
if os.path.exists(TARGET_DIR): shutil.rmtree(TARGET_DIR)
if os.path.exists(TEMP_EXTRACT): shutil.rmtree(TEMP_EXTRACT)
os.makedirs(TARGET_DIR, exist_ok=True)

print('Downloading ChromaDB (~262 MB)...')
subprocess.run(['wget', '-q', '--show-progress', '-O', CHROMA_ZIP, 'https://github.com/arsitekberotot/arsitrad/releases/download/v1.0.0/chroma_db_release.zip'], check=True)

print('Extracting to temporary folder...')
subprocess.run(['unzip', '-q', '-o', CHROMA_ZIP, '-d', TEMP_EXTRACT], check=True)

# Find where the actual sqlite file is (handling nested folders)
sqlite_source_dir = None
for root, dirs, files in os.walk(TEMP_EXTRACT):
    if 'chroma.sqlite3' in files:
        sqlite_source_dir = root
        break

if sqlite_source_dir:
    print(f'Found database in {sqlite_source_dir}. Moving to {TARGET_DIR}...')
    for item in os.listdir(sqlite_source_dir):
        shutil.move(os.path.join(sqlite_source_dir, item), os.path.join(TARGET_DIR, item))
    print('Done!')
else:
    print('Error: chroma.sqlite3 not found in the downloaded ZIP.')

# Cleanup
shutil.rmtree(TEMP_EXTRACT)
if os.path.exists(CHROMA_ZIP): os.remove(CHROMA_ZIP)
print('Final Contents of', TARGET_DIR, ':', os.listdir(TARGET_DIR))

Extracting to temporary folder...
Found database in /content/arsitrad/temp_chroma/chroma_db. Moving to /content/arsitrad/chroma_db...
Done!
Final Contents of /content/arsitrad/chroma_db : ['chroma.sqlite3', 'f74d0699-6baf-4875-ae19-9ad6551fd1d7', 'b3df3f29-b6f3-4249-bc84-194a64429fd7', 'national_embeddings.npy', '0a9eb16f-cb59-4afd-bbc3-d5e2695b65dd', 'national_chunks.json']


## 3. RAG Retrieval Pipeline


In [21]:
import chromadb
from sentence_transformers import SentenceTransformer
import numpy as np
import importlib

# Force a clean state at the library level
importlib.reload(chromadb)
from chromadb.api.shared_system_client import SharedSystemClient
SharedSystemClient.clear_system_cache()

class RegulationRetriever:
    def __init__(self, persist_dir, collection_name):
        self.emb_model = SentenceTransformer(RAG_CONFIG['embedding_model'])
        # Using standard settings to avoid conflict with existing cached systems
        self.client = chromadb.PersistentClient(path=persist_dir)

        try:
            # Try to get existing collection
            self.collection = self.client.get_collection(name=collection_name)
            print(f'✅ Success! Collection "{collection_name}" loaded with {self.collection.count()} chunks.')
        except Exception as e:
            print(f'❌ Failed to get collection: {e}')
            cols = self.client.list_collections()
            print('Available collections:', [c.name for c in cols] if cols else 'None')
            raise e

    def retrieve(self, query, top_k=5, min_similarity=0.3):
        raw_emb = self.emb_model.encode([query])[0]
        q_norm = raw_emb / (np.linalg.norm(raw_emb) + 1e-8)
        results = self.collection.query(
            query_embeddings=[q_norm.tolist()],
            n_results=top_k,
            include=['documents','metadatas','embeddings'],
        )
        outputs = []
        for doc, metadata, emb in zip(results['documents'][0],
                                      results['metadatas'][0],
                                      results['embeddings'][0]):
            if emb is None: continue
            cos_sim = float(np.dot(q_norm, np.array(emb)) / (np.linalg.norm(np.array(emb)) + 1e-8))
            if cos_sim >= min_similarity:
                outputs.append({
                    'text': doc,
                    'source': metadata.get('source','unknown'),
                    'chunk_id': metadata.get('chunk_id',0),
                    'similarity': round(cos_sim,4)
                })
        outputs.sort(key=lambda x: x['similarity'], reverse=True)
        return outputs

    def format_context(self, retrieved):
        if not retrieved:
            return 'No relevant regulations found.'
        parts = []
        for i, item in enumerate(retrieved, 1):
            parts.append(f"[{i}] {item['text']}\n(Source: {item['source']}, chunk {item['chunk_id']})")
        return '\n\n'.join(parts)

# Re-initialize after clearing cache
retriever = RegulationRetriever(str(CHROMA_DIR), RAG_CONFIG['collection_name'])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Success! Collection "arsitrad_national_regulations" loaded with 22649 chunks.


In [15]:
import sqlite3

db_path = '/content/arsitrad/chroma_db/chroma.sqlite3'
conn = sqlite3.connect(db_path)
cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables:", [r[0] for r in cursor.fetchall()])

try:
    cursor2 = conn.execute("SELECT id, name FROM collections;")
    print("Collections:", cursor2.fetchall())
except Exception as e:
    print(f"Error: {e}")
conn.close()

Tables: ['migrations', 'acquire_write', 'collection_metadata', 'segment_metadata', 'tenants', 'databases', 'collections', 'maintenance_log', 'segments', 'embeddings', 'embedding_metadata', 'max_seq_id', 'embedding_fulltext_search', 'embedding_fulltext_search_data', 'embedding_fulltext_search_idx', 'embedding_fulltext_search_content', 'embedding_fulltext_search_docsize', 'embedding_fulltext_search_config', 'embedding_metadata_array', 'embeddings_queue', 'embeddings_queue_config']
Collections: [('3ee3c606-2946-48ee-acea-cf7e9dc8dfa0', 'arsitrad_national_regulations'), ('bab40dd5-19e0-43b9-8ac8-136a3e74bcb3', 'arsitrad_local_regulations')]


In [16]:
!pip install -q --upgrade chromadb
import chromadb
print(f"Updated ChromaDB to version: {chromadb.__version__}")

Updated ChromaDB to version: 1.5.7


In [17]:
import chromadb
import importlib
importlib.reload(chromadb)

# Initialize client with the absolute path
client = chromadb.PersistentClient(path="/content/arsitrad/chroma_db")

try:
    # Attempt to fetch the specific collection
    coll = client.get_collection("arsitrad_national_regulations")
    print(f"✅ Success! Collection 'arsitrad_national_regulations' found.")
    print(f"📊 Total chunks loaded: {coll.count()}")
except Exception as e:
    print(f"❌ Error retrieving collection: {e}")
    print("\nAvailable collections in this database:")
    collections = client.list_collections()
    if not collections:
        print("  (No collections found)")
    for c in collections:
        print(f"  - {c.name}")

❌ Error retrieving collection: Collection [arsitrad_national_regulations] does not exist

Available collections in this database:
  (No collections found)


In [18]:
import os

# Check what's actually in the chroma_db directory
if os.path.exists('/content/arsitrad/chroma_db/'):
    print("Contents of /content/arsitrad/chroma_db/:")
    for f in os.listdir('/content/arsitrad/chroma_db/'):
        file_path = f'/content/arsitrad/chroma_db/{f}'
        size = os.path.getsize(file_path)
        print(f"  {f} — {size/1e6:.1f} MB")
else:
    print("Directory /content/arsitrad/chroma_db/ does not exist.")

Contents of /content/arsitrad/chroma_db/:
  chroma.sqlite3 — 260.4 MB
  f74d0699-6baf-4875-ae19-9ad6551fd1d7 — 0.0 MB
  b3df3f29-b6f3-4249-bc84-194a64429fd7 — 0.0 MB
  national_embeddings.npy — 34.8 MB
  0a9eb16f-cb59-4afd-bbc3-d5e2695b65dd — 0.0 MB
  national_chunks.json — 18.4 MB


In [19]:
import sqlite3
import os

db_path = '/content/arsitrad/chroma_db/chroma.sqlite3'
if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [r[0] for r in cursor.fetchall()]
    print("Tables in ChromaDB:", tables)

    if 'collections' in tables:
        cursor2 = conn.execute("SELECT id, name FROM collections;")
        rows = cursor2.fetchall()
        print(f"Collections found in SQLite: {rows}")
    else:
        print("CRITICAL: 'collections' table missing from SQLite file.")
    conn.close()
else:
    print(f"SQLite file not found at {db_path}")

Tables in ChromaDB: ['migrations', 'acquire_write', 'collection_metadata', 'segment_metadata', 'tenants', 'databases', 'collections', 'maintenance_log', 'segments', 'embeddings', 'embedding_metadata', 'max_seq_id', 'embedding_fulltext_search', 'embedding_fulltext_search_data', 'embedding_fulltext_search_idx', 'embedding_fulltext_search_content', 'embedding_fulltext_search_docsize', 'embedding_fulltext_search_config', 'embedding_metadata_array', 'embeddings_queue', 'embeddings_queue_config']
Collections found in SQLite: [('3ee3c606-2946-48ee-acea-cf7e9dc8dfa0', 'arsitrad_national_regulations'), ('bab40dd5-19e0-43b9-8ac8-136a3e74bcb3', 'arsitrad_local_regulations')]


In [22]:
#@title 3.2 — Test RAG retrieval
test_query = 'syarat teknis ventilasi alami pada rumah tinggal'
results   = retriever.retrieve(test_query, top_k=3)

print(f'Query: {test_query}\n')
for r in results:
    print(f"  [{r['similarity']:.3f}] {r['source']} — {r['text'][:120]}...")
print(f'\nFormatted context:\n{retriever.format_context(results)[:400]}')


Query: syarat teknis ventilasi alami pada rumah tinggal

  [0.828] Permen_21_2021_BGH —  segar ke 
dalam Bangunan Gedung dalam jumlah yang sesuai 
kebutuhan. Selain untuk menyirkulasi gas-gas yang 
berbahaya,...
  [0.680] Permen_26_2008_ProteksiKebakaran — rus sesuai dengan persyaratan teknis untuk 
peralatan memasak komersial, kecuali instalasi tersebut instalasi yang sudah...
  [0.401] SNI_1727_2020_BebanDesainMinimumDanKriteriaTerikat — ondasi
Faktor ketahanan 
harus diberi nilai 0,67 yang diterapkan pada kapasitas ketahanan
untuk penggunaan dengan analis...

Formatted context:
[1]  segar ke 
dalam Bangunan Gedung dalam jumlah yang sesuai 
kebutuhan. Selain untuk menyirkulasi gas-gas yang 
berbahaya, Ventilasi juga digunakan semaksimal mungkin 
untuk meminimalkan beban pendinginan. Sistem ventilasi 
terbagi menjadi dua yaitu sistem ventilasi mekanis dan 
sistem ventilasi alami. Sistem ventilasi mekanis harus 
disediakan apabila sistem ventilasi alami tidak memadai. 
Pere


## 4. Load Fine-tuned Gemma 4 + Inference


In [23]:
#@title 4.1 — Load LoRA adapters with Unsloth
# CRITICAL: If you ran Sections 2-3 first, do Runtime → Restart runtime
# before this cell. Clean GPU state prevents OOM.

import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

lora_path = str(LORA_DIR)

model, _ = FastLanguageModel.from_pretrained(
    model_name     = lora_path,
    max_seq_length = MODEL_CONFIG['max_seq_length'],
    load_in_4bit   = MODEL_CONFIG['load_in_4bit'],
    dtype          = None,
)
FastLanguageModel.for_inference(model)

tokenizer = AutoTokenizer.from_pretrained(lora_path)
print('Model + tokenizer loaded!')


/tmp/ipykernel_3089/2061001445.py:6: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Model + tokenizer loaded!


In [24]:
#@title 4.2 — Build prompt with RAG context
SYSTEM_PROMPT = (
    'Kamu adalah Arsitrad, asisten AI untuk regulasi dan saran arsitektur Indonesia. '
    'Jawablah akurat berdasarkan konteks peraturan yang diberikan. '
    'Jika informasi tidak cukup, katakan secara jujur.'
)

def build_prompt(user_query, context):
    if context and context != 'No relevant regulations found.':
        full_query = f'Konteks peraturan:\n{context}\n\nPertanyaan: {user_query}'
    else:
        full_query = user_query
    return (
        f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
        f'<start_of_turn>user\n{full_query}<end_of_turn>\n'
        f'<start_of_turn>model\n'
    )


In [35]:
def generate_response(user_query, max_new_tokens=256, temperature=0.0):
    retrieved = retriever.retrieve(user_query, top_k=RAG_CONFIG['top_k'])
    context   = retriever.format_context(retrieved)
    prompt    = build_prompt(user_query, context)
    inputs    = tokenizer(prompt, return_tensors='pt', truncation=True).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,   # capped at 150
            temperature=temperature,          # 0.0 = greedy (fastest)
            do_sample=False,                  # no sampling
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    if '<start_of_turn>model\n' in response:
        response = response.split('<start_of_turn>model\n')[-1]
    if '<end_of_turn>' in response:
        response = response.split('<end_of_turn>')[0]
    return response.strip(), context, retrieved

In [36]:
#@title 4.4 — Test inference with sample queries
test_queries = [
    'Apa saja syarat IMB untuk rumah tinggal 2 lantai di Semarang?',
    'Bagaimana klasifikasi kerusakan bangunan akibat gempa bumi?',
    'Strategi passive cooling untuk rumah di daerah tropis basah?',
]

for q in test_queries:
    print(f'\n{"="*60}\nQ: {q}\n{"-"*60}')
    response, context, retrieved = generate_response(q)
    print(f'R: {response[:600]}\n\n[Cited {len(retrieved)} regulation chunks]')


Q: Apa saja syarat IMB untuk rumah tinggal 2 lantai di Semarang?
------------------------------------------------------------
R: <start_of_turn

[Cited 0 regulation chunks]

Q: Bagaimana klasifikasi kerusakan bangunan akibat gempa bumi?
------------------------------------------------------------
R: Klasifikasi kerusakan bangunan akibat gempa bumi dapat diklasifikasikan berdasarkan beberapa aspek utama, yang mencakup:

1. **Kerusakan Struktural (Structural Damage):**
   * **Kerusakan Non-Struktural:** Melibatkan kerusakan pada elemen-elemen pendukung, seperti kolom, balok, dan dinding, yang menyebabkan perubahan pada geometri dan distribusi beban. Kerusakan ini dapat bervariasi dari retak permukaan hingga kegagalan total.
   * **Kerusakan Geometris:** Melibatkan perubahan bentuk fisik bangunan akibat gaya gempa, yang dapat menyebabkan deformasi yang signifikan.
   * **Kerusakan Distribusi

[Cited 5 regulation chunks]

Q: Strategi passive cooling untuk rumah di daerah tropis basah?
---

In [34]:
# Force fastest inference mode
import torch

# Disable gradient tracking globally – not needed for inference
torch.set_grad_enabled(False)

# Verify model is in eval/ inference mode
model.eval()
print(f"Model training mode: {model.training}")
print(f"Inference optimizations applied. Please re-run the following cells.")

Model training mode: False
Inference optimizations applied. Please re-run the following cells.


## 5. Gradio UI — Live Demo


In [28]:
#@title 5.1 — Build & Launch Gradio Interface
import gradio as gr

# Claude-inspired custom CSS
custom_css = """
:root {
    --color-accent: #7C3AED;
    --color-accent-hover: #6D28D9;
    --color-bg: #F8FAFC;
    --color-surface: #FFFFFF;
    --color-border: #E2E8F0;
    --color-text: #1E293B;
    --color-text-muted: #64748B;
    --color-user-bubble: #7C3AED;
    --color-assistant-bubble: #F1F5F9;
    --radius: 12px;
    --font: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
}

body, span, p, div { font-family: var(--font) !important; }

#chat-container { border-radius: var(--radius); overflow: hidden; }

/* Chat messages */
.message.user { background: var(--color-user-bubble) !important; color: white !important; border-radius: 16px 16px 4px 16px; padding: 12px 16px; }
.message.assistant { background: var(--color-assistant-bubble) !important; color: var(--color-text) !important; border-radius: 16px 16px 16px 4px; padding: 12px 16px; }

/* Buttons */
.submit-btn, #submit-btn, button.primary { background: var(--color-accent) !important; border: none !important; border-radius: 8px !important; color: white !important; font-weight: 500 !important; }
.submit-btn:hover, button.primary:hover { background: var(--color-accent-hover) !important; }

/* Textbox */
textarea, input[type=text] { border: 1px solid var(--color-border) !important; border-radius: 10px !important; background: var(--color-surface) !important; color: var(--color-text) !important; font-size: 15px !important; }
textarea:focus, input[type=text]:focus { border-color: var(--color-accent) !important; box-shadow: 0 0 0 3px rgba(124,58,237,0.1) !important; }

/* Title */
h1.title { font-size: 22px !important; font-weight: 700 !important; color: var(--color-text) !important; text-align: center !important; margin-bottom: 4px !important; }
.description { text-align: center !important; color: var(--color-text-muted) !important; font-size: 14px !important; margin-bottom: 20px !important; }

/* Sources box */
.sources-box { background: #F1F5F9; border-left: 3px solid #7C3AED; border-radius: 0 8px 8px 0; padding: 10px 14px; font-size: 13px; color: #64748B; margin-top: 6px; }
.sources-title { font-weight: 600; color: #7C3AED; margin-bottom: 4px; font-size: 12px; text-transform: uppercase; letter-spacing: 0.5px; }

/* Logo header */
.logo-header { text-align: center; padding: 16px 0 8px; }
.logo-text { font-size: 28px; font-weight: 800; background: linear-gradient(135deg, #7C3AED, #EC4899); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
.logo-sub { font-size: 13px; color: #64748B; margin-top: 4px; }

/* Scrollbar */
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: transparent; }
::-webkit-scrollbar-thumb { background: #CBD5E1; border-radius: 3px; }
"""

def arsitrad_chat(message, history):
    response, context, retrieved = generate_response(message)
    sources = '\n'.join([f"- [{r['similarity']:.3f}] {r['source']}" for r in retrieved])
    full = f"{response}\n\n<div class='sources-box'><div class='sources-title'>ሒ Sources</div>{sources}</div>"
    return full

# Build interface with custom theme
with gr.Blocks(css=custom_css, title='Arsitrad — AI Konsultan Hukum Bangunan Indonesia') as demo:
    # Header
    gr.HTML('<div class="logo-header"><div class="logo-text">ARSITRAD</div><div class="logo-sub">AI Konsultan Regulasi &amp; Arsitektur Indonesia</div></div>')
    
    gr.ChatInterface(
        fn=arsitrad_chat,
        title='',
        description='Gemma 4 fine-tuned · RAG on 22.649 chunks regulasi konstruksi Indonesia · Modul: Bencana, IMB, Permukiman, Pendinginan',
        examples=[
            ['Rumah rusak akibat gempa di Cianjur, dinding retak diagonal 45°, atap bergeser — apa yang harus dilakukan?'],
            ['Apa syarat obtaining Izin Mendirikan Bangunan (IMB) di Indonesia?'],
            ['Jelaskan strategi passive cooling untuk bangunan di iklim tropis Indonesia.'],
            ['Bagaimana prosedur upgrading permukiman kumuh sesuai regulasi?'],
        ],
        chatbot=gr.Chatbot(
            height=480,
            show_copy_button=True,
            avatar_images=(None, None),
            render=False,
        ),
        type='messages',
    )

demo.launch(server_name='0.0.0.0', server_port=7860, share=True)


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c31c01143e56277196.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Troubleshooting

| Problem | Fix |
|---|---|
| OOM on T4 | Runtime → Restart runtime before inference (clears GPU memory from downloads) |
| Model not loading | Use LoRA path directly, or `google/gemma-4-E2B-it` for base model |
| ChromaDB empty | Re-run Section 2.3 — wget may timeout |
| Slow retrieval | Reduce `top_k` in RAG_CONFIG (e.g. 3 instead of 5) |
| Gradio link not working | Wait 30s after launch for Colab to provision the URL |
